In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [3]:
import json


with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Region Extraction from S3 Bucket ARN\n\nHere's a Python function that extracts the AWS region from an S3 bucket ARN:\n\n```python\ndef extract_region_from_s3_arn(arn: str) -> str | None:\n    \"\"\"\n    Extract the AWS region from an S3 bucket ARN string.\n    \n    Args:\n        arn: S3 bucket ARN string (e.g., 'arn:aws:s3:::my-bucket-us-east-1')\n    \n    Returns:\n        The AWS region if found, None otherwise\n    \n    Examples:\n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket-us-east-1')\n        'us-east-1'\n        \n        >>> extract_region_from_s3_arn('arn:aws:s3:::my-bucket')\n        None\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    # S3 bucket ARN format: arn:aws:s3:::bucket-name\n    # Regions are typically appended to bucket name: bucket-name-region\n    # AWS regions follow pattern: region-code (e.g., us-east-1, eu-west-1)\n    \n    # Split by colons to get bucket name part\n   